# HMM-DFC — Chapter 6 reference implementation

Regenerates **every table and figure** of Chapter 6 from one end-to-end execution, implementing
the methodology exactly:

- **Temporal partitioning** into $M=12$ calendar streams (daily → monthly resampling; node $m$
  holds all month-$m$ observations across years, ≈60 per node).
- **Hybrid z-score/IQR normalisation** (Eq. hybrid_normalisation).
- **Per-node FCM** (Eqs. fcm_membership_dfc / fcm_centroid_dfc; $m_f=2$, $\varepsilon=10^{-4}$)
  and **per-node Gaussian HMM** with FCM-initialised, membership-weighted emissions
  (Eq. hmm_emission), trained by Baum–Welch.
- **EBNR** (Eqs. node_entropy / entropy_weight). ⚠ *Direction note:* the equation assigns
  **low weight to high-entropy (noisy/under-sampled) nodes** — this implementation follows the
  equation; the chapter's intuition paragraph must be corrected to match (supervisor point 6).
- **Entropy-weighted aggregation** with Hungarian state alignment across nodes.
- **VATR** — ⚠ implemented in the **corrected off-diagonal form with row renormalisation**
  (supervisor point 7): applied uniformly to all $(i,j)$ as originally written, the row-constant
  factor cancels in Viterbi and the normalised forward pass and the refinement is inert. Here,
  rising volatility reallocates transition mass from self-transitions toward regime changes.
- **Baselines:** centralised HMM, GARCH(1,1), distributed-FCM-only, and the VATR/EBNR ablation
  ladder. **Metrics:** FPC, fuzzy silhouette, P/R/F1 for high-volatility regime detection
  (rolling-std 75th-percentile ground truth), forecast MSE/MAE, runtime-vs-clusters,
  CPU/memory-vs-volume, node-failure robustness, streaming latency.

**Outputs** (into `OUTDIR`, filenames matching the chapter's `\includegraphics` paths exactly,
typos included): `runtime_numberofClusters.png`, `Centralized vs Distributed Models.png`,
`robust_node_failiure.png`, `abilation_study_fcm_hmm.png`, `tab_model_comparison.tex`,
`tab_forecasting.tex`, `tab_ablation.tex`, `tab_latency.tex`, `results_summary_ch6.json`.

**Provenance discipline** (same as Chapter 5): the run you keep becomes the chapter's single
source of truth — regenerate table values *and* prose numbers from `results_summary_ch6.json`;
the published SMC values came from unavailable code and are superseded. **Feature note:** Yahoo's
daily API carries no dividend yield / EPS; without a CSV supplying them the feature set is
OHLCV-derived and §6.4.1's bullet list should be trimmed to match.

## 0. Run configuration

In [1]:
# ----------------------- run configuration -----------------------
OUTDIR = "figures/chapter6"
DATA_DIR = None      # folder with GSPC_daily.csv (Date, OHLCV [, DividendYield, EPS]) or None
SYNTHETIC = False    # True -> synthetic OHLCV smoke test (no network needed)

## 1. Imports and hyperparameters

In [2]:
import json
import os
import time
import tracemalloc
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import linear_sum_assignment, minimize

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RED, GREY, GREEN, BLUE, ORANGE = "#c0392b", "#95a5a6", "#2e8b57", "#2e6da4", "#e07b39"

M_NODES = 12
C_DEFAULT = 3
M_FUZZ = 2.0
BETA = 0.5
EPS_FCM = 1e-4
SEED = 42

## 2. Data: loading, hybrid normalisation, monthly features, ground truth

In [3]:
def load_prices(data_dir=None, synthetic=False):
    if synthetic:
        return synth_ohlcv()
    if data_dir:
        p = os.path.join(data_dir, "GSPC_daily.csv")
        if os.path.exists(p):
            df = pd.read_csv(p, parse_dates=["Date"]).set_index("Date")
            return df
    import yfinance as yf
    df = yf.download("^GSPC", start="1965-01-01", end="2024-12-31",
                     auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    return df[["Open", "High", "Low", "Close", "Volume"]].dropna()


def synth_ohlcv(n_years=60, seed=SEED):
    """Regime-switching GARCH price path with OHLCV, for smoke tests."""
    rng = np.random.default_rng(seed)
    T = n_years * 252
    params = [(1e-6, 0.05, 0.90), (4e-6, 0.10, 0.85), (2e-5, 0.15, 0.70)]
    seg = np.zeros(T, int)
    t = 0
    while t < T:
        k = rng.integers(0, 3)
        L = rng.integers(200, 900)
        seg[t:t + L] = k
        t += L
    r = np.zeros(T)
    s2 = 1e-4
    for t in range(T):
        w, a, b = params[seg[t]]
        s2 = w + a * (r[t - 1] ** 2 if t else 0) + b * s2
        r[t] = np.sqrt(s2) * rng.standard_normal()
    close = 100 * np.exp(np.cumsum(r))
    high = close * np.exp(np.abs(rng.normal(0, 0.004, T)))
    low = close * np.exp(-np.abs(rng.normal(0, 0.004, T)))
    open_ = np.roll(close, 1); open_[0] = close[0]
    vol = np.exp(14 + 0.5 * rng.standard_normal(T) + 20 * np.abs(r))
    idx = pd.bdate_range("1965-01-04", periods=T)
    return pd.DataFrame({"Open": open_, "High": high, "Low": low,
                         "Close": close, "Volume": vol}, index=idx)


def hybrid_normalise(X):
    """Eq. (hybrid_normalisation): 0.5 z-score + 0.5 robust (median/IQR)."""
    mu, sd = X.mean(0), X.std(0)
    med = np.median(X, 0)
    iqr = np.subtract(*np.percentile(X, [75, 25], axis=0))
    sd = np.where(sd < 1e-12, 1, sd)
    iqr = np.where(iqr < 1e-12, 1, iqr)
    return 0.5 * (X - mu) / sd + 0.5 * (X - med) / iqr


def build_monthly(df):
    """Monthly-resampled feature matrix + auxiliary series.
    Features per month: log return, mean daily |return| (realised vol),
    mean intramonth range, log volume change [, DividendYield, EPS if given].
    """
    d = df.copy()
    d["lr"] = np.log(d["Close"] / d["Close"].shift(1))
    d["rng"] = (d["High"] - d["Low"]) / d["Close"]
    g = d.resample("ME")
    feat = pd.DataFrame({
        "ret": g["lr"].sum(),
        "rv": g["lr"].std(),
        "range": g["rng"].mean(),
        "dvol": np.log(g["Volume"].mean()).diff(),
    })
    for extra in ("DividendYield", "EPS"):
        if extra in d.columns:
            feat[extra.lower()] = g[extra].mean()
    feat = feat.replace([np.inf, -np.inf], np.nan)
    feat["dvol"] = feat["dvol"].fillna(0.0)   # zero/missing early volume -> flat
    feat = feat.dropna()
    X = hybrid_normalise(feat.values.astype(float))
    months = feat.index.month.values          # calendar stream id (1..12)
    rv_raw = feat["rv"].values                # realised vol (target basis)
    return X, months, rv_raw, feat.index


def ground_truth_regimes(rv_raw, window=6, q=0.75):
    """High-volatility regime labels: rolling std of the vol series
    thresholded at its 75th percentile (chapter, Evaluation Metrics)."""
    s = pd.Series(rv_raw).rolling(window, min_periods=1).std()
    thr = np.nanquantile(s, q)
    return (s.values > thr).astype(int)

## 3. Fuzzy C-Means (custom NumPy) + clustering-quality metrics

In [4]:
class FCM:
    def __init__(self, C, m_f=M_FUZZ, eps=EPS_FCM, max_iter=300, seed=SEED):
        self.C, self.m_f, self.eps, self.max_iter = C, m_f, eps, max_iter
        self.rng = np.random.default_rng(seed)

    def fit(self, X, init_centres=None):
        n, d = X.shape
        if init_centres is None:
            idx = self.rng.choice(n, self.C, replace=False)
            self.centres = X[idx] + 1e-6 * self.rng.standard_normal((self.C, d))
        else:
            self.centres = init_centres.copy()
        for _ in range(self.max_iter):
            U = self._memberships(X, self.centres)
            Um = U ** self.m_f
            new = (Um.T @ X) / np.maximum(Um.sum(0)[:, None], 1e-12)
            new = np.nan_to_num(new, nan=0.0, posinf=0.0, neginf=0.0)
            if np.linalg.norm(new - self.centres) < self.eps:
                self.centres = new
                break
            self.centres = new
        self.U = self._memberships(X, self.centres)
        return self

    def _memberships(self, X, centres):
        D = np.maximum(
            np.linalg.norm(X[:, None, :] - centres[None, :, :], axis=2), 1e-12)
        p = 2.0 / (self.m_f - 1.0)
        R = (D[:, :, None] / D[:, None, :]) ** p       # (n, C, C)
        U = 1.0 / R.sum(axis=2)
        return U / U.sum(1, keepdims=True)

    def memberships(self, X):
        return self._memberships(X, self.centres)


def fpc(U):
    return float((U ** 2).sum() / len(U))


def fuzzy_silhouette(X, U):
    """Silhouette on the defuzzified partition, membership-weighted."""
    lab = U.argmax(1)
    if len(np.unique(lab)) < 2:
        return 0.0
    from sklearn.metrics import silhouette_samples
    s = silhouette_samples(X, lab)
    w = U.max(1)
    return float((s * w).sum() / w.sum())

## 4. Gaussian HMM (diagonal covariance) — Baum–Welch, FCM-initialised emissions

In [5]:
class GaussianHMM:
    def __init__(self, C, seed=SEED):
        self.C = C
        self.rng = np.random.default_rng(seed)

    def init_from_fcm(self, X, fcm: FCM):
        """Emissions from the fuzzy partition (Eq. hmm_emission):
        mean = centroid; covariance membership-weighted."""
        U = fcm.memberships(X)
        self.mu = fcm.centres.copy()
        d = X.shape[1]
        self.var = np.zeros((self.C, d))
        for j in range(self.C):
            w = U[:, j]
            diff = X - self.mu[j]
            self.var[j] = np.maximum(
                (w[:, None] * diff ** 2).sum(0) / max(w.sum(), 1e-9), 1e-4)
        self.A = np.full((self.C, self.C), 0.1 / max(self.C - 1, 1))
        np.fill_diagonal(self.A, 0.9)
        self.pi = np.ones(self.C) / self.C
        return self

    def _logB(self, X):
        lb = np.zeros((len(X), self.C))
        for j in range(self.C):
            lb[:, j] = (-0.5 * np.log(2 * np.pi * self.var[j]).sum()
                        - 0.5 * ((X - self.mu[j]) ** 2 / self.var[j]).sum(1))
        return lb

    def fit(self, X, n_iter=25, update_emissions=True):
        for _ in range(n_iter):
            logB = self._logB(X)
            B = np.exp(logB - logB.max(1, keepdims=True))
            N = len(X)
            alpha = np.zeros((N, self.C)); c = np.zeros(N)
            alpha[0] = self.pi * B[0]; c[0] = alpha[0].sum() + 1e-300
            alpha[0] /= c[0]
            for t in range(1, N):
                alpha[t] = (alpha[t - 1] @ self.A) * B[t]
                c[t] = alpha[t].sum() + 1e-300
                alpha[t] /= c[t]
            beta = np.ones((N, self.C))
            for t in range(N - 2, -1, -1):
                beta[t] = (self.A @ (B[t + 1] * beta[t + 1])) / c[t + 1]
            g = alpha * beta
            g /= np.maximum(g.sum(1, keepdims=True), 1e-300)
            xi = np.zeros((self.C, self.C))
            for t in range(N - 1):
                xi += np.outer(alpha[t], B[t + 1] * beta[t + 1]) * self.A / c[t + 1]
            self.A = xi / np.maximum(xi.sum(1, keepdims=True), 1e-12)
            self.pi = g[0]
            if update_emissions:
                w = g.sum(0)
                self.mu = (g.T @ X) / np.maximum(w[:, None], 1e-12)
                for j in range(self.C):
                    diff = X - self.mu[j]
                    self.var[j] = np.maximum(
                        (g[:, j][:, None] * diff ** 2).sum(0)
                        / max(w[j], 1e-9), 1e-4)
        self.gamma_ = g
        return self

    def posteriors(self, X, A_seq=None):
        """Filtered-smoothed state posteriors; A_seq optionally provides a
        time-varying transition matrix (VATR)."""
        logB = self._logB(X)
        B = np.exp(logB - logB.max(1, keepdims=True))
        N = len(X)
        alpha = np.zeros((N, self.C)); c = np.zeros(N)
        alpha[0] = self.pi * B[0]; c[0] = alpha[0].sum() + 1e-300
        alpha[0] /= c[0]
        for t in range(1, N):
            A = self.A if A_seq is None else A_seq[t]
            alpha[t] = (alpha[t - 1] @ A) * B[t]
            c[t] = alpha[t].sum() + 1e-300
            alpha[t] /= c[t]
        return alpha

    def viterbi(self, X, A_seq=None):
        logB = self._logB(X)
        N = len(X)
        logd = np.zeros((N, self.C))
        back = np.zeros((N, self.C), int)
        logd[0] = np.log(self.pi + 1e-300) + logB[0]
        for t in range(1, N):
            A = self.A if A_seq is None else A_seq[t]
            logA = np.log(A + 1e-300)
            m = logd[t - 1][:, None] + logA
            back[t] = m.argmax(0)
            logd[t] = m.max(0) + logB[t]
        z = np.zeros(N, int)
        z[-1] = logd[-1].argmax()
        for t in range(N - 2, -1, -1):
            z[t] = back[t + 1, z[t + 1]]
        return z

## 5. HMM-DFC core — nodes, EBNR, aggregation, VATR
State alignment across nodes uses Hungarian matching on centroids so aggregation averages like
with like. `use_vatr` / `use_ebnr` flags drive the ablation ladder. VATR is the corrected
off-diagonal form (see header note).

In [6]:
def align_states(centres_ref, centres_m):
    """Hungarian matching of node states to reference states by centroid
    distance, so aggregation averages like with like."""
    D = np.linalg.norm(centres_ref[:, None, :] - centres_m[None, :, :], axis=2)
    D = np.nan_to_num(D, nan=1e9, posinf=1e9, neginf=1e9)
    r, c = linear_sum_assignment(D)
    perm = np.empty(len(r), int)
    perm[r] = c
    return perm


def train_node(Xm, C, seed):
    fcm = FCM(C, seed=seed).fit(Xm)
    hmm = GaussianHMM(C, seed=seed).init_from_fcm(Xm, fcm)
    hmm.fit(Xm, n_iter=20, update_emissions=False)   # emissions anchored to FCM
    return fcm, hmm


class HMMDFC:
    def __init__(self, C=C_DEFAULT, beta=BETA, use_vatr=True, use_ebnr=True,
                 seed=SEED, n_workers=M_NODES):
        self.C, self.beta = C, beta
        self.use_vatr, self.use_ebnr = use_vatr, use_ebnr
        self.seed, self.n_workers = seed, n_workers

    # ---- distributed fit ---------------------------------------------------
    def fit(self, X, months, parallel=True):
        all_nodes = [X[months == m] for m in range(1, M_NODES + 1)]
        active = [i for i, Xm in enumerate(all_nodes) if len(Xm) >= 3 * self.C]
        self.nodes_data = [all_nodes[i] for i in active]
        seeds = [self.seed + i for i in active]
        t0 = time.perf_counter()
        if parallel:
            with ThreadPoolExecutor(max_workers=self.n_workers) as ex:
                out = list(ex.map(lambda a: train_node(a[0], self.C, a[1]),
                                  zip(self.nodes_data, seeds)))
        else:
            out = [train_node(a, self.C, s)
                   for a, s in zip(self.nodes_data, seeds)]
        self.fit_time = time.perf_counter() - t0
        self.fcms = [o[0] for o in out]
        self.hmms = [o[1] for o in out]

        # EBNR weights (Eqs. node_entropy / entropy_weight)
        ws = []
        for fcm, Xm in zip(self.fcms, self.nodes_data):
            U = np.clip(fcm.U, 1e-12, 1)
            H = float(-(U * np.log(U)).sum())
            w = 1 - H / (len(Xm) * np.log(self.C))
            ws.append(max(w, 1e-6))
        ws = np.asarray(ws)
        self.ebnr_raw = ws.copy()
        if not self.use_ebnr:
            ws = np.ones(len(self.nodes_data))
        self.w = ws / ws.sum()

        # state alignment to node 0, then entropy-weighted aggregation
        ref = self.fcms[0].centres
        C_agg = np.zeros_like(ref)
        A_agg = np.zeros((self.C, self.C))
        var_agg = np.zeros_like(self.hmms[0].var)
        for wm, fcm, hmm in zip(self.w, self.fcms, self.hmms):
            perm = align_states(ref, fcm.centres)
            C_agg += wm * fcm.centres[perm]
            A_agg += wm * hmm.A[np.ix_(perm, perm)]
            var_agg += wm * hmm.var[perm]
        A_agg /= np.maximum(A_agg.sum(1, keepdims=True), 1e-12)
        self.c_global, self.A_global, self.var_global = C_agg, A_agg, var_agg

        # global inference model
        self.g = GaussianHMM(self.C, seed=self.seed)
        self.g.mu, self.g.var = self.c_global, self.var_global
        self.g.A, self.g.pi = self.A_global, np.ones(self.C) / self.C
        # order states by volatility (feature 1 = rv) for interpretation
        order = np.argsort(self.g.mu[:, 1])
        self.g.mu, self.g.var = self.g.mu[order], self.g.var[order]
        self.g.A = self.g.A[np.ix_(order, order)]
        self.c_global = self.g.mu
        return self

    # ---- VATR (Eqs. volatility_shift / vatr_adjustment) --------------------
    def vatr_A_seq(self, X):
        fcm_g = FCM(self.C, seed=self.seed)
        fcm_g.centres = self.c_global
        U = fcm_g.memberships(X)
        D2 = ((X[:, None, :] - self.c_global[None]) ** 2).sum(2)
        A_seq = np.repeat(self.A_global[None], len(X), axis=0)
        if not self.use_vatr:
            return A_seq
        offdiag = ~np.eye(self.C, dtype=bool)
        for t in range(1, len(X)):
            dV = float((U[t] * (D2[t] - D2[t - 1])).sum())
            A = self.A_global.copy()
            # NOTE (methodology correction): the multiplier is applied to
            # OFF-DIAGONAL transitions only, then rows are renormalised.
            # Applied uniformly to all (i, j) as in the chapter's Eq. as
            # written, the row-constant factor cancels in both Viterbi and
            # the normalised forward recursion and the refinement is inert;
            # the off-diagonal form realises the stated intent ("boost
            # transition probabilities, encouraging regime changes").
            A[offdiag] = A[offdiag] * (1 + self.beta * np.tanh(dV))
            A = np.maximum(A, 1e-9)
            A_seq[t] = A / A.sum(1, keepdims=True)
        return A_seq

    def infer(self, X):
        A_seq = self.vatr_A_seq(X)
        z = self.g.viterbi(X, A_seq)
        post = self.g.posteriors(X, A_seq)
        return z, post

## 6. Baselines — centralised HMM, distributed-FCM-only, GARCH(1,1)

In [7]:
def centralised_hmm(X, C, seed=SEED):
    fcm = FCM(C, seed=seed).fit(X)
    hmm = GaussianHMM(C, seed=seed).init_from_fcm(X, fcm)
    t0 = time.perf_counter()
    hmm.fit(X, n_iter=25)
    fit_time = time.perf_counter() - t0
    order = np.argsort(hmm.mu[:, 1])
    hmm.mu, hmm.var = hmm.mu[order], hmm.var[order]
    hmm.A = hmm.A[np.ix_(order, order)]
    return hmm, fit_time


def distributed_fcm_only(X, months, C, seed=SEED):
    fcms = [FCM(C, seed=seed + m).fit(X[months == m])
            for m in range(1, M_NODES + 1)]
    ref = fcms[0].centres
    Cg = np.zeros_like(ref)
    for f in fcms:
        Cg += f.centres[align_states(ref, f.centres)]
    Cg /= M_NODES
    Cg = Cg[np.argsort(Cg[:, 1])]
    fg = FCM(C, seed=seed)
    fg.centres = Cg
    return fg


def fit_garch(r):
    r = r - r.mean()
    v = np.var(r)

    def nll(p):
        w, a, b = p
        if w <= 0 or a < 0 or b < 0 or a + b >= 0.999:
            return 1e10
        s2 = np.empty(len(r)); s2[0] = v
        for t in range(1, len(r)):
            s2[t] = w + a * r[t - 1] ** 2 + b * s2[t - 1]
        s2 = np.maximum(s2, 1e-12)
        return 0.5 * np.sum(np.log(s2) + r ** 2 / s2)

    res = minimize(nll, np.array([0.05 * v, 0.05, 0.9]), method="Nelder-Mead",
                   options={"maxiter": 1500})
    w, a, b = res.x
    s2 = np.empty(len(r)); s2[0] = v
    for t in range(1, len(r)):
        s2[t] = max(w + a * r[t - 1] ** 2 + b * s2[t - 1], 1e-12)
    return np.sqrt(s2)

## 7. Metrics — P/R/F1, regime flags, volatility forecasts (leak-free, lagged posteriors)

In [8]:
def prf(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    p = tp / max(tp + fp, 1)
    r = tp / max(tp + fn, 1)
    f = 2 * p * r / max(p + r, 1e-12)
    return p, r, f


def high_vol_pred_from_states(z, post, mu_rv, q=0.75):
    """Base-rate-aligned high-volatility decision: the model's continuous
    high-vol score is the posterior probability of the highest-volatility
    state, thresholded at the score's own q-quantile. This matches every
    detector's flagged fraction to the ground truth's construction (rolling
    std > its 75th percentile), so F1 compares ranking quality rather than
    base-rate agreement. (An argmax-state rule flags at the state-occupancy
    rate instead, structurally penalising HMM detectors against
    threshold-style baselines such as GARCH.)"""
    top = int(np.argmax(mu_rv))
    score = post[:, top]
    thr = np.quantile(score, q)
    return (score > thr).astype(int)


def vol_forecast_from_post(post, state_rv):
    """One-step vol forecast: E[rv | filtered posterior], lagged one step
    (posterior at t predicts t+1, no look-ahead)."""
    exp_rv = post @ state_rv
    return np.roll(exp_rv, 1)[1:]           # predict t from info at t-1

## 8. Experiments — all tables and figures
Model comparison, forecasting, runtime-vs-clusters, CPU/memory-vs-volume, node-failure
robustness (averaged over 5 random dropout draws per failure fraction), VATR/EBNR ablation,
and streaming latency on a daily-data batch replay.

In [ ]:
def tex_write(path, s):
    with open(path, "w") as f:
        f.write(s)


def run_all(outdir, data_dir=None, synthetic=False):
    os.makedirs(outdir, exist_ok=True)
    RES = {}
    rng = np.random.default_rng(SEED)

    print("[data] loading…")
    df = load_prices(data_dir, synthetic)
    X, months, rv_raw, idx = build_monthly(df)
    y_true = ground_truth_regimes(rv_raw)
    print(f"  monthly observations: {len(X)}  (per node ≈ {len(X)//12})")

    # ------------------------------------------------ main models
    print("[1/6] model comparison…")
    dfc = HMMDFC(C=C_DEFAULT).fit(X, months)
    z, post = dfc.infer(X)
    state_rv = np.array([rv_raw[z == j].mean() if (z == j).any()
                         else rv_raw.mean() for j in range(C_DEFAULT)])
    mu_rv_feature = dfc.g.mu[:, 1]

    ch, ch_time = centralised_hmm(X, C_DEFAULT)
    zc = ch.viterbi(X)
    postc = ch.posteriors(X)

    fg = distributed_fcm_only(X, months, C_DEFAULT)
    Ug = fg.memberships(X)
    zf = Ug.argmax(1)

    garch_vol_d = fit_garch(np.log(df["Close"] / df["Close"].shift(1)).dropna().values)
    garch_vol = pd.Series(garch_vol_d,
                          index=df.index[1:]).resample("ME").mean()
    garch_vol = garch_vol.reindex(idx).ffill().bfill().values
    y_garch = (garch_vol > np.quantile(garch_vol, 0.75)).astype(int)

    def row(name, U, ztags, Xs=X):
        p, r, f = prf(y_true, ztags)
        return dict(name=name, FPC=fpc(U), Sil=fuzzy_silhouette(Xs, U),
                    P=p, R=r, F1=f)

    U_dfc = FCM(C_DEFAULT, seed=SEED)
    U_dfc.centres = dfc.c_global
    U_dfc = U_dfc.memberships(X)
    U_ch = postc                                        # HMM posteriors as soft partition
    pg, rg, fg1 = prf(y_true, y_garch)
    rows = [
        dict(name="GARCH(1,1)", FPC=None, Sil=None, P=pg, R=rg, F1=fg1),
        row("Distributed FCM only", Ug,
            (Ug[:, int(np.argmax(fg.centres[:, 1]))]
             > np.quantile(Ug[:, int(np.argmax(fg.centres[:, 1]))], 0.75)
             ).astype(int)),
        row("Centralised HMM", U_ch,
            high_vol_pred_from_states(zc, postc, ch.mu[:, 1])),
        row("HMM-DFC (full)", U_dfc,
            high_vol_pred_from_states(z, post, mu_rv_feature)),
    ]
    RES["model_comparison"] = rows
    RES["ebnr_weights"] = {"normalised": dfc.w.tolist(),
                           "max_over_uniform": float(dfc.w.max() * len(dfc.w)),
                           "raw": dfc.ebnr_raw.tolist()}
    L = [r"\begin{tabular}{lccccc}", r"\toprule",
         r"\textbf{Model} & \textbf{FPC} & \textbf{Silhouette} & "
         r"\textbf{Precision} & \textbf{Recall} & \textbf{F1} \\", r"\midrule"]
    for rr in rows:
        nm = r"\textbf{HMM-DFC (full)}" if "full" in rr["name"] else rr["name"]
        bold = "full" in rr["name"]
        fmt = (lambda v: f"\\textbf{{{v:.3f}}}") if bold else (lambda v: f"{v:.3f}")
        f2 = (lambda v: f"\\textbf{{{v:.2f}}}") if bold else (lambda v: f"{v:.2f}")
        c1 = "---" if rr['FPC'] is None else fmt(rr['FPC'])
        c2 = "---" if rr['Sil'] is None else f2(rr['Sil'])
        L.append(f"{nm} & {c1} & {c2} & "
                 f"{f2(rr['P'])} & {f2(rr['R'])} & {f2(rr['F1'])} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]
    tex_write(os.path.join(outdir, "tab_model_comparison.tex"), "\n".join(L))

    # ------------------------------------------------ forecasting
    print("[2/6] forecasting…")
    y_vol = rv_raw[1:]
    preds = {
        "HMM-DFC (full)": vol_forecast_from_post(post, state_rv),
        "Centralised HMM": vol_forecast_from_post(
            postc, np.array([rv_raw[zc == j].mean() if (zc == j).any()
                             else rv_raw.mean() for j in range(C_DEFAULT)])),
        "Distributed FCM only": vol_forecast_from_post(
            Ug, np.array([rv_raw[zf == j].mean() if (zf == j).any()
                          else rv_raw.mean() for j in range(C_DEFAULT)])),
        "GARCH(1,1)": garch_vol[:-1],
    }
    frows = []
    for nm, pv in preds.items():
        pv = np.asarray(pv)[: len(y_vol)]
        e = pv - y_vol
        frows.append(dict(name=nm, MSE=float(np.mean(e ** 2)),
                          MAE=float(np.mean(np.abs(e)))))
    frows.sort(key=lambda d: d["MSE"])
    RES["forecasting"] = frows
    L = [r"\begin{tabular}{lcc}", r"\toprule",
         r"\textbf{Model} & \textbf{MSE} & \textbf{MAE} \\", r"\midrule"]
    for rr in frows:
        nm = r"\textbf{HMM-DFC (full)}" if "full" in rr["name"] else rr["name"]
        L.append(f"{nm} & ${rr['MSE']:.3e}$ & ${rr['MAE']:.3e}$ \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]
    tex_write(os.path.join(outdir, "tab_forecasting.tex"), "\n".join(L))

    # ------------------------------------------------ scalability: runtime vs C
    print("[3/6] runtime vs clusters…")
    Cs = list(range(2, 9))
    t_dist, t_cent = [], []
    for Cx in Cs:
        d = HMMDFC(C=Cx).fit(X, months)
        t_dist.append(d.fit_time)
        _, tc = centralised_hmm(X, Cx)
        t_cent.append(tc)
    RES["runtime_vs_clusters"] = {"C": Cs, "distributed": t_dist,
                                  "centralised": t_cent}
    fig, ax = plt.subplots(figsize=(7, 4.4))
    ax.plot(Cs, t_cent, "-o", c=GREY, label="Centralised HMM")
    ax.plot(Cs, t_dist, "-o", c=GREEN, label="HMM-DFC (12 nodes, parallel)")
    ax.set_xlabel("Number of clusters $C$")
    ax.set_ylabel("Training runtime (s)")
    ax.set_title("Runtime versus number of clusters")
    ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "runtime_numberofClusters.png"), dpi=200)
    plt.close(fig)

    # ------------------------------------------------ resource usage vs volume
    print("[4/6] CPU/memory vs data volume…")
    scales = [1, 2, 4, 8]
    mem_c, mem_d, cpu_c, cpu_d = [], [], [], []
    for s in scales:
        Xs = np.tile(X, (s, 1)) + 1e-6 * rng.standard_normal((len(X) * s, X.shape[1]))
        ms = np.tile(months, s)
        tracemalloc.start()
        t0 = time.perf_counter()
        centralised_hmm(Xs, C_DEFAULT)
        cpu_c.append(time.perf_counter() - t0)
        mem_c.append(tracemalloc.get_traced_memory()[1] / 1e6)
        tracemalloc.stop()
        tracemalloc.start()
        t0 = time.perf_counter()
        HMMDFC(C=C_DEFAULT).fit(Xs, ms)
        cpu_d.append(time.perf_counter() - t0)
        mem_d.append(tracemalloc.get_traced_memory()[1] / 1e6 / M_NODES)  # per node
        tracemalloc.stop()
    RES["resources"] = {"scale": scales, "cpu_centralised": cpu_c,
                        "cpu_distributed": cpu_d, "mem_centralised_MB": mem_c,
                        "mem_per_node_MB": mem_d}
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
    axes[0].plot(scales, cpu_c, "-o", c=GREY, label="Centralised")
    axes[0].plot(scales, cpu_d, "-o", c=GREEN, label="HMM-DFC (wall-clock)")
    axes[0].set_xlabel("Data volume (×)"); axes[0].set_ylabel("CPU time (s)")
    axes[0].set_title("CPU usage"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(scales, mem_c, "-o", c=GREY, label="Centralised (total)")
    axes[1].plot(scales, mem_d, "-o", c=GREEN, label="HMM-DFC (per node)")
    axes[1].set_xlabel("Data volume (×)"); axes[1].set_ylabel("Peak memory (MB)")
    axes[1].set_title("Memory usage"); axes[1].legend(); axes[1].grid(alpha=0.3)
    fig.suptitle("Centralised vs distributed resource consumption")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "Centralized vs Distributed Models.png"),
                dpi=200)
    plt.close(fig)

    # ------------------------------------------------ fault tolerance
    print("[5/6] node-failure robustness…")
    fracs = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    f1s = []
    for fr in fracs:
        vals = []
        for rep in range(5):                     # average over random dropouts
            keep = rng.random(M_NODES) >= fr
            if not keep.any():
                keep[rng.integers(0, M_NODES)] = True
            d = HMMDFC(C=C_DEFAULT, seed=SEED + rep)
            mkeep = np.isin(months, np.where(keep)[0] + 1)
            d.fit(X[mkeep], months[mkeep])
            zz, pp = d.infer(X)
            pred = high_vol_pred_from_states(zz, pp, d.g.mu[:, 1])
            vals.append(prf(y_true, pred)[2])
        f1s.append(float(np.mean(vals)))
    RES["fault_tolerance"] = {"failure_frac": fracs, "F1": f1s}
    fig, ax = plt.subplots(figsize=(7, 4.4))
    ax.plot([f * 100 for f in fracs], f1s, "-o", c=GREEN)
    ax.set_xlabel("Node failure (%)"); ax.set_ylabel("Regime-detection F1")
    ax.set_title("Robustness to node failures")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "robust_node_failiure.png"), dpi=200)
    plt.close(fig)

    # ------------------------------------------------ ablation
    print("[6/6] ablation + streaming latency…")
    abl_rows = []
    for name, uv, ue in [("HMM-DFC (no VATR/EBNR)", False, False),
                         ("+ VATR only", True, False),
                         ("+ VATR + EBNR (full)", True, True)]:
        d = HMMDFC(C=C_DEFAULT, use_vatr=uv, use_ebnr=ue).fit(X, months)
        zz, pp = d.infer(X)
        Ux = FCM(C_DEFAULT, seed=SEED); Ux.centres = d.c_global
        Ux = Ux.memberships(X)
        p, r, f = prf(y_true, high_vol_pred_from_states(zz, pp, d.g.mu[:, 1]))
        abl_rows.append(dict(name=name, FPC=fpc(Ux),
                             Sil=fuzzy_silhouette(X, Ux), P=p, R=r, F1=f))
    RES["ablation"] = abl_rows
    L = [r"\begin{tabular}{lccccc}", r"\toprule",
         r"\textbf{Configuration} & \textbf{FPC} & \textbf{Silhouette} & "
         r"\textbf{Precision} & \textbf{Recall} & \textbf{F1} \\", r"\midrule"]
    for i, rr in enumerate(abl_rows):
        bold = i == 2
        fmt = (lambda v: f"\\textbf{{{v:.3f}}}") if bold else (lambda v: f"{v:.3f}")
        f2 = (lambda v: f"\\textbf{{{v:.2f}}}") if bold else (lambda v: f"{v:.2f}")
        L.append(f"{rr['name']} & {fmt(rr['FPC'])} & {f2(rr['Sil'])} & "
                 f"{f2(rr['P'])} & {f2(rr['R'])} & {f2(rr['F1'])} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]
    tex_write(os.path.join(outdir, "tab_ablation.tex"), "\n".join(L))
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    xs = np.arange(3)
    ax.bar(xs - 0.15, [r_["FPC"] for r_ in abl_rows], 0.3, color=BLUE, label="FPC")
    ax.bar(xs + 0.15, [r_["F1"] for r_ in abl_rows], 0.3, color=GREEN, label="F1")
    ax.set_xticks(xs)
    ax.set_xticklabels(["no VATR/EBNR", "+ VATR", "+ VATR + EBNR"], fontsize=9)
    ax.set_title("Ablation: VATR and EBNR contributions")
    ax.legend(); ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "abilation_study_fcm_hmm.png"), dpi=200)
    plt.close(fig)

    # streaming latency (batch-stream on daily data replay, per chapter)
    d_daily = df.copy()
    d_daily["lr"] = np.log(d_daily["Close"] / d_daily["Close"].shift(1))
    d_daily["rng"] = (d_daily["High"] - d_daily["Low"]) / d_daily["Close"]
    d_daily["dvol"] = np.log(d_daily["Volume"].replace(0, np.nan)).diff()
    d_daily["dvol"] = d_daily["dvol"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    d_daily["rv"] = d_daily["lr"].rolling(20).std()
    Xd = d_daily[["lr", "rv", "rng", "dvol"]].replace(
        [np.inf, -np.inf], np.nan).dropna().values
    Xd = hybrid_normalise(Xd)
    batches = np.array_split(Xd[-2520:], 24)          # ~monthly batches, 10y
    t_fcm = t_hmm = t_vatr = t_sync = 0.0
    dS = HMMDFC(C=C_DEFAULT).fit(X, months)
    node_fcms = [FCM(C_DEFAULT, seed=SEED + i) for i in range(M_NODES)]
    for f_ in node_fcms:
        f_.centres = dS.c_global.copy()
    for bi, B in enumerate(batches):
        nid = bi % M_NODES
        t0 = time.perf_counter()
        node_fcms[nid].fit(B, init_centres=node_fcms[nid].centres)
        t_fcm += time.perf_counter() - t0
        t0 = time.perf_counter()
        h = GaussianHMM(C_DEFAULT, seed=SEED).init_from_fcm(B, node_fcms[nid])
        h.fit(B, n_iter=5, update_emissions=False)
        t_hmm += time.perf_counter() - t0
        t0 = time.perf_counter()
        dS.vatr_A_seq(B)
        t_vatr += time.perf_counter() - t0
        t0 = time.perf_counter()
        # EBNR sync across current node states
        U = np.clip(node_fcms[nid].U, 1e-12, 1)
        _ = 1 - float(-(U * np.log(U)).sum()) / (len(B) * np.log(C_DEFAULT))
        ref = dS.c_global
        dS.c_global = 0.9 * dS.c_global + 0.1 * node_fcms[nid].centres[
            align_states(ref, node_fcms[nid].centres)]
        t_sync += time.perf_counter() - t0
    nB = len(batches)
    lat = {"Local FCM update": t_fcm / nB, "Local HMM update": t_hmm / nB,
           "VATR adjustment": t_vatr / nB,
           "Global EBNR synchronisation": t_sync / nB}
    lat["Total per batch"] = sum(lat.values())
    t0 = time.perf_counter()
    centralised_hmm(Xd[-2520:], C_DEFAULT)
    lat["Centralised HMM (full refit)"] = time.perf_counter() - t0
    RES["latency_seconds"] = lat
    L = [r"\begin{tabular}{lc}", r"\toprule",
         r"\textbf{Component} & \textbf{Time (s)} \\", r"\midrule"]
    fmt_s = lambda v: "$<0.001$" if v < 5e-4 else f"{v:.3f}"
    for k in ["Local FCM update", "Local HMM update", "VATR adjustment",
              "Global EBNR synchronisation"]:
        L.append(f"{k} & {fmt_s(lat[k])} \\\\")
    L += [r"\midrule",
          f"\\textbf{{Total per batch}} & \\textbf{{{lat['Total per batch']:.3f}}} \\\\",
          f"Centralised HMM (full refit) & {lat['Centralised HMM (full refit)']:.3f} \\\\",
    tex_write(os.path.join(outdir, "tab_latency.tex"), "\n".join(L))

    with open(os.path.join(outdir, "results_summary_ch6.json"), "w") as f:
        json.dump(RES, f, indent=2, default=float)
    print(f"\nAll Chapter 6 outputs written to {outdir}/")
    return RES

## 9. Execute

In [10]:
RES = run_all(OUTDIR, DATA_DIR, SYNTHETIC)

[data] loading…


$^GSPC: possibly delisted; no price data found  (1d 1965-01-01 -> 2024-12-31)

1 Failed download:
['^GSPC']: possibly delisted; no price data found  (1d 1965-01-01 -> 2024-12-31)


IndexError: index -1 is out of bounds for axis 0 with size 0

## 10. Key results at a glance

In [ ]:
import pandas as pd
print("MODEL COMPARISON");   display(pd.DataFrame(RES["model_comparison"]).set_index("name").round(3))
print("FORECASTING");        display(pd.DataFrame(RES["forecasting"]).set_index("name"))
print("ABLATION");           display(pd.DataFrame(RES["ablation"]).set_index("name").round(3))
print("FAULT TOLERANCE");    display(pd.DataFrame(RES["fault_tolerance"]))
print("LATENCY (s)");        display(pd.Series(RES["latency_seconds"]).round(3))

## 11. Figures inline

In [ ]:
from IPython.display import Image, display
for fp in ["runtime_numberofClusters.png", "Centralized vs Distributed Models.png",
           "robust_node_failiure.png", "abilation_study_fcm_hmm.png"]:
    p = os.path.join(OUTDIR, fp)
    if os.path.exists(p):
        print(fp); display(Image(p, width=760))

---
**Updating the chapter.** Copy the `tab_*.tex` bodies into the corresponding table environments,
drop the four PNGs into `chapter6/` (names already match the `\includegraphics` paths), and
update every prose number — F1/FPC/silhouette values, the speedup and latency ratios, the
fault-tolerance percentages — from `results_summary_ch6.json`. Apply the Eq. (6.12) off-diagonal
correction and the EBNR intuition fix in the text **before** archiving this run, so code and
chapter state the same mathematics. Archive this notebook + JSON + figures in one tagged commit.